# Training YOLOv8: Koin / Korek Api / Tutup Botol

Jalankan sel dari atas ke bawah. **Sebelum mulai:** `Runtime` > `Change runtime type` > **T4 GPU** > Save.

Persiapan di Google Drive (`My Drive`):
- `sortir-yolo.zip` (zip folder proyek ini)
- *(opsional)* zip dataset korek api hasil unduh manual, mis. `korek_api.zip` (lihat Langkah 5)
- *(opsional)* folder `foto_sendiri_mentah` berisi foto sendiri per kelas (lihat Langkah 7)

## 1. Cek GPU

In [ ]:
!nvidia-smi

## 2. Ambil proyek dari Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/sortir-yolo
!unzip -q -o /content/drive/MyDrive/sortir-yolo.zip -d /content
%cd /content/sortir-yolo
!ls

## 3. Install library

In [ ]:
!pip install -q -r requirements.txt

## 4. Isi API key Roboflow

Cara mengambil API key: login di app.roboflow.com > Settings workspace > **API Keys** > salin *Private API Key*.

Di Colab simpan lewat menu **Secrets** (ikon kunci di sidebar kiri): tambah secret bernama `ROBOFLOW_API_KEY`, isi dengan key kamu, lalu aktifkan *Notebook access*. Setelah itu jalankan sel di bawah.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
    print("API key terbaca dari Secrets.")
except Exception:
    import getpass
    os.environ["ROBOFLOW_API_KEY"] = getpass.getpass("Tempel API key Roboflow: ")

## 5. Unduh dataset lewat API

- **koin** (object detection) dan **tutup botol** (klasifikasi) akan diunduh otomatis.
- **korek api**: project Universe-nya belum punya versi, jadi KEMUNGKINAN GAGAL lewat API. Itu normal. Solusinya (sekali saja): buka halaman datasetnya > **Fork Dataset** > di workspace kamu klik **Generate** > **Download Dataset** > format **YOLOv8** > unduh zip > taruh di Drive sebagai `korek_api.zip`, lalu jalankan sel *Korek manual* di bawah.

In [ ]:
!python 01_download_datasets.py

In [ ]:
# Korek manual (jalankan hanya kalau korek_api gagal di atas)
!python 01_download_datasets.py --only korek_api --zip korek_api=/content/drive/MyDrive/korek_api.zip

## 6. Beri kotak otomatis pada dataset tutup botol

Dataset tutup botol tidak punya kotak, jadi dibuatkan otomatis dengan YOLO-World (prompt: "bottle cap"). **Cek gambar preview di bawah**: kotak merah harus pas di tutup botol. Kalau banyak yang salah, jangan lanjut training dulu.

In [ ]:
!python 02_autolabel_tutup_botol.py --method world

In [ ]:
from IPython.display import Image, display
display(Image('outputs/preview_autolabel_tutup_botol.jpg'))

## 7. (Disarankan) Tambahkan foto sendiri

Ambil foto di laptop dengan `python extra_capture_data.py`, simpan folder `datasets_raw/foto_sendiri_mentah` ke Drive, lalu jalankan sel ini. Lewati sel ini kalau belum punya fotonya.

In [ ]:
import os
src = '/content/drive/MyDrive/foto_sendiri_mentah'
if os.path.isdir(src):
    !mkdir -p datasets_raw
    !cp -r "{src}" datasets_raw/foto_sendiri_mentah
    !python extra_autolabel_own.py
    display(Image('outputs/preview_autolabel_foto_sendiri.jpg'))
else:
    print('Folder foto_sendiri_mentah tidak ada di Drive -> dilewati.')

## 8. Gabungkan dataset (3 kelas) dan lihat laporannya

In [ ]:
!python 03_merge_datasets.py

Baca bagian **CATATAN** di laporan: jumlah data per kelas, kebocoran data, dan kelas yang datanya terlalu sedikit.

## 9. Training

Hasil disimpan ke Google Drive (`MyDrive/yolo_runs/sortir`) supaya aman kalau sesi terputus. Kalau terputus: jalankan ulang Langkah 2-3 lalu sel *Lanjutkan training*.

In [ ]:
!python 04_train.py --epochs 80 --batch 16 --model yolov8n.pt \
    --project /content/drive/MyDrive/yolo_runs --name sortir

In [ ]:
# Lanjutkan training yang terputus (jalankan HANYA kalau perlu)
# !python 04_train.py --resume --project /content/drive/MyDrive/yolo_runs --name sortir

## 10. Lihat hasil

In [ ]:
from IPython.display import Image, display
base = '/content/drive/MyDrive/yolo_runs/sortir'
for f in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    p = f'{base}/{f}'
    if os.path.exists(p):
        display(Image(p))

## 11. Ambil model

`best.pt` sudah ada di Drive: `MyDrive/yolo_runs/sortir/weights/best.pt`. Sel di bawah mengunduhnya ke laptop. Taruh di folder `models/` proyek, lalu jalankan `python 05_detect.py`.

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/yolo_runs/sortir/weights/best.pt')